In [1]:
import torch
import numpy as np
from bhcr_architecture import BHCRSudokuSolver, neural_guided_bitmask_finish, is_board_valid
from dataset_pipeline import create_initial_candidate_mask


e:\projects\python code snippets\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def solve_string_puzzle(puzzle_str: str):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 1. Create model and load your trained weights
    model = BHCRSudokuSolver(hidden_dim=96, max_iterations=10).to(device)
    model.load_state_dict(torch.load("bhcr_sudoku_solver.pt", map_location=device))
    model.eval()
    print("Model loaded successfully!")

    # 2. Clean the input string
    clean_str = puzzle_str.strip().replace(' ', '').replace('\n', '').replace('.', '0')
    if len(clean_str) != 81:
        raise ValueError(f"Puzzle must be exactly 81 characters. Got {len(clean_str)}")

    quiz = np.fromiter((int(ch) if ch in '123456789' else 0 for ch in clean_str),
                       dtype=np.int64, count=81).reshape(9, 9)

    # 3. Create the initial candidate mask (very important for your architecture)
    mask = create_initial_candidate_mask(quiz)
    mask_tensor = torch.from_numpy(mask).unsqueeze(0).to(device)

    # 4. Run the model
    import time
    start = time.perf_counter()
    with torch.no_grad(), torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
        step_logits = model(mask_tensor, early_exit_inference=True)
        final_logits = step_logits[-1].float().cpu().numpy()[0]

    # 5. Finish with the neural-guided bitmask solver (gives 100% accuracy)
    solved_zero_idx = neural_guided_bitmask_finish(quiz, final_logits)
    elapsed_ms = (time.perf_counter() - start) * 1000

    solved = solved_zero_idx + 1   # convert to 1-9

    if is_board_valid(solved_zero_idx):
        print(f"\n[Success] Solved in {elapsed_ms:.2f} ms")
    else:
        print(f"\n[Warning] Verification failed")

    # Pretty print
    print("\nSolved Board:")
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("-" * 21)
        row = ""
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row += "| "
            row += f"{solved[r, c]} "
        print(row)

    return solved


In [7]:
test_puzzle = """809000010010083000000500020000037208000000000205000943000070000480000030000908460"""
    
print("Solving custom puzzle...")
solve_string_puzzle(test_puzzle)

Solving custom puzzle...
Using device: cuda
Model loaded successfully!

[Success] Solved in 39.85 ms

Solved Board:
8 6 9 | 7 4 2 | 3 1 5 
5 1 2 | 6 8 3 | 7 9 4 
7 3 4 | 5 1 9 | 8 2 6 
---------------------
6 9 1 | 4 3 7 | 2 5 8 
3 4 8 | 2 9 5 | 6 7 1 
2 7 5 | 8 6 1 | 9 4 3 
---------------------
9 5 6 | 3 7 4 | 1 8 2 
4 8 7 | 1 2 6 | 5 3 9 
1 2 3 | 9 5 8 | 4 6 7 


array([[8, 6, 9, 7, 4, 2, 3, 1, 5],
       [5, 1, 2, 6, 8, 3, 7, 9, 4],
       [7, 3, 4, 5, 1, 9, 8, 2, 6],
       [6, 9, 1, 4, 3, 7, 2, 5, 8],
       [3, 4, 8, 2, 9, 5, 6, 7, 1],
       [2, 7, 5, 8, 6, 1, 9, 4, 3],
       [9, 5, 6, 3, 7, 4, 1, 8, 2],
       [4, 8, 7, 1, 2, 6, 5, 3, 9],
       [1, 2, 3, 9, 5, 8, 4, 6, 7]])